# CareTrack AI - Disease Prediction Model Training

Complete ML training pipeline following the 11-step specification:
1. Import libraries
2. Load dataset
3. Preprocess (one-hot, LabelEncode target)
4. Train/test split (80/20, stratify)
5. Select models (NB, DT, RF, SVM, GradientBoosting)
6. Train with timing
7. Evaluate (accuracy, precision, recall, F1)
8. Classification report
9. Confusion matrix heatmap
10. Save best model
11. Prediction function test

## Step 1: Import Required Libraries

In [ ]:
import os
import json
import time
import warnings

# Data Processing
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

# Machine Learning
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix
)
from sklearn.preprocessing import LabelEncoder

# Algorithms
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC

# Model Persistence
import joblib

warnings.filterwarnings('ignore')
print('All libraries imported successfully!')

## Step 2: Load the Dataset

In [ ]:
DATASET_PATH = os.path.join('Dataset', 'data.csv')
MODEL_DIR = 'Model'
os.makedirs(MODEL_DIR, exist_ok=True)

df = pd.read_csv(DATASET_PATH)
print(f'Dataset shape: {df.shape}')
print(f'Columns: {df.shape[1]} ({df.shape[1] - 1} features + 1 target)')
print(f'Rows: {df.shape[0]:,}')
print(f'Target column: {df.columns[0]!r}')
print(f'Unique diseases: {df.iloc[:, 0].nunique()}')
df.head()

## Step 3: Preprocess the Symptom Data

In [ ]:
# Clean column names
df.columns = df.columns.str.strip()

target_col = df.columns[0]
feature_cols = list(df.columns[1:])

# Verify one-hot encoded structure
feature_values = df[feature_cols].values
unique_vals = np.unique(feature_values)
print(f'Unique feature values: {unique_vals}')
assert set(unique_vals).issubset({0, 1}), 'Features should be binary (0/1)!'
print('Confirmed: one-hot encoded (Structure A)')

# Encode target column
le = LabelEncoder()
df['target_encoded'] = le.fit_transform(df[target_col].str.strip())
print(f'LabelEncoder: {len(le.classes_)} disease classes')

# Separate features and target
X = df[feature_cols].values
y = df['target_encoded'].values
print(f'Features X: {X.shape}')
print(f'Target y: {y.shape}')

## Step 4: Split the Dataset (80/20, Stratified)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f'Training set: {X_train.shape[0]:,} samples')
print(f'Testing set:  {X_test.shape[0]:,} samples')

## Step 5: Select Models

Symptom datasets are highly categorical and sparse, making these models ideal:
- **Naive Bayes**: Fast baseline for text-based probabilities
- **Decision Tree**: Maps yes/no symptom pathways
- **Random Forest**: Handles complex symptom combinations
- **SVM (SVC)**: Excellent for high-dimensional categorical spaces
- **Gradient Boosting**: State-of-the-art tabular performance

In [ ]:
# SVM subsample (SVC is O(n^2) -- too slow on 200K+ rows)
SVM_SAMPLE_SIZE = 30000
np.random.seed(42)
if X_train.shape[0] > SVM_SAMPLE_SIZE:
    svm_idx = np.random.choice(X_train.shape[0], SVM_SAMPLE_SIZE, replace=False)
    X_train_svm, y_train_svm = X_train[svm_idx], y_train[svm_idx]
    print(f'SVM subsample: {SVM_SAMPLE_SIZE:,} rows')
else:
    X_train_svm, y_train_svm = X_train, y_train

models = {
    'GaussianNB': {
        'model': GaussianNB(),
        'X_train': X_train, 'y_train': y_train,
    },
    'DecisionTree': {
        'model': DecisionTreeClassifier(random_state=42, max_depth=30),
        'X_train': X_train, 'y_train': y_train,
    },
    'RandomForest': {
        'model': RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1, max_depth=30),
        'X_train': X_train, 'y_train': y_train,
    },
    'SVM (SVC)': {
        'model': SVC(kernel='rbf', probability=True, random_state=42),
        'X_train': X_train_svm, 'y_train': y_train_svm,
    },
    'GradientBoosting': {
        'model': GradientBoostingClassifier(n_estimators=100, max_depth=5, learning_rate=0.1, random_state=42, subsample=0.8),
        'X_train': X_train, 'y_train': y_train,
    },
}
print(f'Models to train: {list(models.keys())}')

## Steps 6, 7 & 8: Train, Evaluate & Classification Reports

In [ ]:
results = {}

for name, cfg in models.items():
    print(f'\n{"=" * 60}')
    print(f'Training: {name}')
    print(f'{"=" * 60}')
    
    model = cfg['model']
    xt, yt = cfg['X_train'], cfg['y_train']
    
    # Train with timing
    start = time.time()
    model.fit(xt, yt)
    train_time = time.time() - start
    print(f'  Training time: {train_time:.2f}s')
    
    # Evaluate
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, average='weighted', zero_division=0)
    rec = recall_score(y_test, y_pred, average='weighted', zero_division=0)
    f1 = f1_score(y_test, y_pred, average='weighted', zero_division=0)
    
    results[name] = {
        'model': model,
        'accuracy': round(acc, 4),
        'precision': round(prec, 4),
        'recall': round(rec, 4),
        'f1_score': round(f1, 4),
        'train_time_seconds': round(train_time, 2),
        'y_pred': y_pred,
    }
    
    print(f'  Accuracy:  {acc:.4f}')
    print(f'  Precision: {prec:.4f}')
    print(f'  Recall:    {rec:.4f}')
    print(f'  F1-Score:  {f1:.4f}')
    
    # Classification Report (truncated for readability)
    report = classification_report(y_test, y_pred, target_names=le.classes_, zero_division=0)
    lines = report.strip().split('\n')
    if len(lines) > 40:
        print('\n  Classification Report (truncated):')
        for l in lines[:15]: print(f'  {l}')
        print(f'  ... ({len(lines) - 30} more classes) ...')
        for l in lines[-15:]: print(f'  {l}')
    else:
        print(f'\n  Classification Report:')
        for l in lines: print(f'  {l}')

## Model Comparison Chart

In [ ]:
# Compare all models visually
model_names = list(results.keys())
metrics = ['accuracy', 'precision', 'recall', 'f1_score']

fig, axes = plt.subplots(1, 4, figsize=(18, 5))
colors = ['#4338CA', '#7C3AED', '#0891B2', '#DC2626', '#059669']

for i, metric in enumerate(metrics):
    values = [results[n][metric] for n in model_names]
    bars = axes[i].bar(model_names, values, color=colors)
    axes[i].set_title(metric.replace('_', ' ').title(), fontsize=12, fontweight='bold')
    axes[i].set_ylim(0, 1.05)
    axes[i].tick_params(axis='x', rotation=45)
    for bar, val in zip(bars, values):
        axes[i].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.01,
                     f'{val:.3f}', ha='center', va='bottom', fontsize=9)

plt.suptitle('Model Comparison - All Metrics', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(MODEL_DIR, 'model_comparison.png'), dpi=150, bbox_inches='tight')
plt.show()

## Step 9: Confusion Matrix for Best Model

In [ ]:
# Find best model
best_name = max(results, key=lambda n: results[n]['f1_score'])
best_f1 = results[best_name]['f1_score']

print('Model Rankings by F1-Score:')
print('-' * 60)
for name in sorted(results, key=lambda n: results[n]['f1_score'], reverse=True):
    r = results[name]
    marker = ' <-- BEST' if name == best_name else ''
    print(f"  {name:20s}  F1={r['f1_score']:.4f}  Acc={r['accuracy']:.4f}  Time={r['train_time_seconds']:.2f}s{marker}")

# Confusion matrix
y_pred_best = results[best_name]['y_pred']
cm = confusion_matrix(y_test, y_pred_best)

num_classes = len(le.classes_)
if num_classes > 30:
    class_counts = np.bincount(y_test)
    top_indices = np.argsort(class_counts)[-20:]
    mask = np.isin(y_test, top_indices)
    cm_small = confusion_matrix(y_test[mask], y_pred_best[mask], labels=top_indices)
    labels = le.classes_[top_indices]
    suffix = ' (Top 20 Diseases)'
else:
    cm_small, labels, suffix = cm, le.classes_, ''

fig, ax = plt.subplots(figsize=(14, 12))
sns.heatmap(cm_small, annot=(cm_small.shape[0] <= 25), fmt='d', cmap='Blues',
            xticklabels=labels, yticklabels=labels, ax=ax, linewidths=0.5)
ax.set_xlabel('Predicted Disease', fontsize=12)
ax.set_ylabel('Actual Disease', fontsize=12)
ax.set_title(f'Confusion Matrix - {best_name}{suffix}', fontsize=14, fontweight='bold')
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.yticks(fontsize=8)
plt.tight_layout()
plt.savefig(os.path.join(MODEL_DIR, 'confusion_matrix.png'), dpi=150, bbox_inches='tight')
plt.show()

## Step 10: Save the Best Performing Model

In [ ]:
best_model = results[best_name]['model']

# Save model artifacts
joblib.dump(best_model, os.path.join(MODEL_DIR, 'best_model.pkl'))
joblib.dump(le, os.path.join(MODEL_DIR, 'label_encoder.pkl'))
joblib.dump(feature_cols, os.path.join(MODEL_DIR, 'feature_columns.pkl'))

print(f'Saved best_model.pkl ({best_name})')
print(f'Saved label_encoder.pkl ({len(le.classes_)} classes)')
print(f'Saved feature_columns.pkl ({len(feature_cols)} features)')

# Save training results JSON
results_json = {}
for name, res in results.items():
    results_json[name] = {
        'accuracy': res['accuracy'],
        'precision': res['precision'],
        'recall': res['recall'],
        'f1_score': res['f1_score'],
        'train_time_seconds': res['train_time_seconds'],
    }
results_json['best_model'] = best_name
results_json['num_diseases'] = len(le.classes_)
results_json['num_features'] = len(feature_cols)
results_json['num_train_samples'] = int(X_train.shape[0])
results_json['num_test_samples'] = int(X_test.shape[0])
results_json['disease_classes'] = list(le.classes_)

with open(os.path.join(MODEL_DIR, 'training_results.json'), 'w') as f:
    json.dump(results_json, f, indent=2)
print('Saved training_results.json')
print(f'\nAll artifacts saved to: {os.path.abspath(MODEL_DIR)}')

## Step 11: Test Prediction Function

In [ ]:
from predict import predict_disease

# Test with sample symptoms
test_symptoms = ['fever', 'cough', 'sore_throat', 'headache', 'fatigue']
result = predict_disease(test_symptoms, top_n=5)

print(f'Test symptoms: {test_symptoms}')
print(f'Matched: {result["symptoms_matched"]}')
print(f'Unmatched: {result["symptoms_unmatched"]}')
print(f'Active features: {result["active_features"]}')
print(f'Model: {result["model_name"]}')
print(f'\nTop {len(result["predictions"])} Predictions:')
print('-' * 50)
for p in result['predictions']:
    print(f"  {p['disease']:40s}  {p['confidence']:.2f}%")

---

## Training Complete!

The best model has been saved to `Model/`. The Flask backend (`app.py`) will load this pre-trained model automatically -- no need to retrain.

**Files saved:**
- `Model/best_model.pkl` - The trained model
- `Model/label_encoder.pkl` - Disease name encoder
- `Model/feature_columns.pkl` - Feature column names
- `Model/training_results.json` - All metrics and comparison
- `Model/confusion_matrix.png` - Confusion matrix heatmap
- `Model/model_comparison.png` - Model comparison chart